# Pairwise Quality Assessment - Attribute Analysis

Analyze how each method performs on different user-labeled quality attributes.

In [ ]:
# Imports
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd()))

from sim_bench.analysis.utils import get_project_root
from merge_pairwise_results import load_pairwise_results, merge_pairwise_results
from analyze_pairwise_by_attributes import (
    merge_results_with_labels,
    compute_accuracy_by_attribute,
    compute_accuracy_by_label_presence,
    plot_accuracy_by_majority_label,
    plot_method_performance_by_label,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print("✓ Imports successful")

## 1. Load Data

In [ ]:
# Configuration
PROJECT_ROOT = get_project_root()
# BENCHMARK_DIR = PROJECT_ROOT / "outputs" / "pairwise_benchmark_3hour" / "pairwise_20251120_100520"
BENCHMARK_DIR = PROJECT_ROOT / "outputs" / "pairwise_benchmark_test" / "pairwise_20251125_004908"
MERGED_RESULTS_PATH = BENCHMARK_DIR / "merged_pairwise_results.csv"
LABELS_PATH = Path(r"D:\Similar Images\automatic_triage_photo_series\photo_triage_pairs_keyword_labels.csv")

print(f"Benchmark dir: {BENCHMARK_DIR}")
print(f"Results file exists: {MERGED_RESULTS_PATH.exists()}")
print(f"Labels file exists: {LABELS_PATH.exists()}")

In [ ]:
# Load pairwise results
pairwise_merged = pd.read_csv(MERGED_RESULTS_PATH)
print(f"Loaded pairwise results: {pairwise_merged.shape}")
print(f"Columns: {list(pairwise_merged.columns[:10])}...")
pairwise_merged.head()

In [ ]:
df_labels = pd.read_csv(LABELS_PATH)
print(df_labels.columns)
df_labels.head()

In [ ]:
df_labels[['compareFile1', 'compareFile2', 'majority_label']]

In [ ]:
df_reviews = pd.read_csv(r'D:\Similar Images\automatic_triage_photo_series\reviews_df.csv')
df_reviews.head()
df_reviews.dropna(subset=["reason_text"], axis=0).reset_index(drop=True).drop(columns=['userChoice','label']
).to_csv((r'D:\Similar Images\automatic_triage_photo_series\reviews_df_cleaned.csv'))

In [ ]:
df_reviews['label'].unique()

In [ ]:
df_full_labels = pd.read_csv(r'D:\Similar Images\automatic_triage_photo_series\train_val\labels_db.csv')
print(df_full_labels.columns)

df_full_labels.head()

In [ ]:
df_labels['majority_label'].value_counts()

In [ ]:
# Merge with user attribute labels
combined_df = merge_results_with_labels(MERGED_RESULTS_PATH, LABELS_PATH)

# Get method names
pairwise_methods = [col.replace('_correct', '') for col in combined_df.columns if col.endswith('_correct')]

print(f"\nMerged data shape: {combined_df.shape}")
print(f"Pairs with labels: {len(combined_df)} ({len(combined_df)/len(pairwise_merged)*100:.1f}% coverage)")
print(f"\nMethods: {', '.join(pairwise_methods)}")

## 2. Accuracy by Majority Label

Show how each method performs on different attribute categories (composition, sharpness, exposure, etc.)

In [ ]:
# Compute accuracy by majority label
majority_label_accuracy = compute_accuracy_by_attribute(combined_df, pairwise_methods, 'majority_label')

print("Method Accuracy by User-Labeled Attribute:")
print("="*80)
display(majority_label_accuracy.sort_values('count', ascending=False))

In [ ]:
# Visualize as heatmap
fig = plot_accuracy_by_majority_label(majority_label_accuracy, pairwise_methods, figsize=(14, 8))
plt.show()

## 3. Attribute Presence Analysis

Compare performance when specific attributes are mentioned vs not mentioned

In [ ]:
# Performance when specific attributes mentioned
label_cols = [col for col in combined_df.columns if col.startswith('label_') and 
              col not in ['label_no_reason_given', 'label_no_specific_attribute']]

label_presence = compute_accuracy_by_label_presence(combined_df, pairwise_methods, label_cols)

print("Accuracy when attribute PRESENT vs ABSENT:")
print("="*80)
display(label_presence.sort_values('present_count', ascending=False))

## 4. Composition Deep-Dive

In [ ]:
# Composition analysis
composition_pairs = combined_df[combined_df['label_composition'] > 0]
print(f"COMPOSITION ANALYSIS ({len(composition_pairs)} pairs where 'composition' mentioned)")
print("="*80)

for method in pairwise_methods:
    correct_col = f'{method}_correct'
    comp_acc = composition_pairs[correct_col].mean()
    overall_acc = combined_df[correct_col].mean()
    diff = comp_acc - overall_acc
    
    sign = "✓" if diff > 0 else "✗" if diff < 0 else "="
    print(f"{sign} {method:25s}: {comp_acc:.3f} (overall: {overall_acc:.3f}, diff: {diff:+.3f})")

## 5. Per-Method Visualizations

Detailed analysis for top methods

In [ ]:
# Visualize top 3 methods
for method in ['Combined-RuleBased', 'Sharpness', 'Colorfulness']:
    print(f"\n{'='*80}")
    print(f"{method} - Performance by Attribute")
    print(f"{'='*80}")
    fig = plot_method_performance_by_label(label_presence, method, figsize=(14, 6))
    if fig:
        plt.show()
    else:
        print(f"Could not create visualization for {method}")